# Membrane Protein Data - Data Cleaning

## Objectives

* Load and inspect the membrane protein dataset obtained from the OPM database.
* Assess the structure and quality of the dataset.
* Identify missing values, duplicate records, and inappropriate data types.
* Clean and transform the data where necessary, documenting the reason for each decision.
* Prepare a cleaned dataset for exploratory data analysis.

## Inputs

'data/proteins-2026-08-10.csv` - raw membrane protein structural data downloaded from the OPM database.

## Outputs
'data/cleaned/proteins_cleaned.csv' - cleaned membrane protein dataset prepared for exploratory data analysis. 

Additional Comments

The raw dataset will be preserved unchanged.
Cleaning decisions will be based on the relevance and quality of individual variables rather than automatically removing all records containing missing values.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os

project_dir = r"C:\Users\trasn\OneDrive\Dokumente\programing\37 week-app\membrane-protein-data-analysis"

os.chdir(project_dir)

print(os.getcwd())

C:\Users\trasn\OneDrive\Dokumente\programing\37 week-app\membrane-protein-data-analysis


# Section 1
Data Collection and Initial Inspection


The raw OPM membrane protein dataset will first be loaded and inspected to understand its dimensions, variables, data types, and overall structure before any cleaning decisions are made.

## Load the Dataset

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/proteins-2026-08-10.csv")
df.head()


,id,ordering,family_name_cache,species_name_cache,membrane_name_cache,name,description,comments,pdbid,resolution,...,superfamily_id,classtype_id,type_id,secondary_representations_count,structure_subunits_count,citations_count,created_at,updated_at,uniprotcode,interpro
0,1,6024.0,OmpA family,Escherichia coli,Gram-neg. outer,"Outer membrane protein A (OMPA), disordered loops",NaN,OmpA is required for the action of colicins K ...,"=""1qjp""",1.65,...,26,2,1,3,1,2,2018-08-13 03:49:46 UTC,2023-02-22 20:43:56 UTC,OMPA_ECOLI,NaN
1,2,6027.0,Enterobacterial Ail/Lom protein,Escherichia coli,Gram-neg. outer,Outer membrane protein X (OMPX),NaN,OmpX from Escherichia coli promotes adhesion t...,"=""1qj8""",1.9,...,26,2,1,7,1,1,2018-08-13 03:49:46 UTC,2023-02-22 20:43:57 UTC,OMPX_ECOLI,NaN
2,3,6032.0,Opacity porins,Neisseria meningitidis,Gram-neg. outer,Outer membrane protein NspA,NaN,Pathogenic Neisseria spp. possess a repertoire...,"=""1p4t""",2.55,...,235,2,1,0,1,0,2018-08-13 03:49:46 UTC,2023-02-22 20:43:57 UTC,Q9RP17_NEIME,NaN
3,4,5624.0,Influenza virus matrix protein 2,Influenza A virus,Viral,"M2 proton channel of Influenza A, closed state...",NaN,NaN,"=""3lbw""",1.65,...,185,11,1,3,4,0,2018-08-13 03:49:46 UTC,2023-02-22 20:43:54 UTC,M2_I97A1,NaN
4,5,6047.0,"OM protease omptin, OMPT",Yersinia pestis,Gram-neg. outer,Plasminogen activator PLA (coagulase/fibrinoly...,NaN,NaN,"=""2x55""",1.85,...,27,2,1,2,1,0,2018-08-13 03:49:46 UTC,2023-02-22 20:43:56 UTC,COLY_YERPE,NaN


---

## Dataset Dimensions

Before cleaning the data, the size of the dataset is checked to understand how many observations and variables are available for analysis.

In [3]:
df.shape

(8915, 33)

The raw dataset contains 8,915 observations and 33 variables. This provides a sufficiently large dataset for exploring patterns in the structural and biological characteristics of membrane proteins.

## Column Names

The column names are inspected to understand the variables available in the dataset and to identify which may be relevant to the membrane protein analysis.

In [4]:
df.columns

Index(['id', 'ordering', 'family_name_cache', 'species_name_cache',
       'membrane_name_cache', 'name', 'description', 'comments', 'pdbid',
       'resolution', 'topology_subunit', 'topology_show_in', 'thickness',
       'thicknesserror', 'subunit_segments', 'tilt', 'tilterror', 'gibbs',
       'tau', 'verification', 'membrane_id', 'species_id', 'family_id',
       'superfamily_id', 'classtype_id', 'type_id',
       'secondary_representations_count', 'structure_subunits_count',
       'citations_count', 'created_at', 'updated_at', 'uniprotcode',
       'interpro'],
      dtype='object')

### Initial observation

The dataset contains 33 variables representing a mixture of protein identifiers, biological classifications, structural measurements, database relationships, and metadata.

Variables such as `resolution`, `thickness`, `subunit_segments`, `tilt`, and `gibbs` appear particularly relevant for structural analysis, while `family_name_cache`, `species_name_cache`, and `membrane_name_cache` may allow comparisons between biological groups.

Other columns, such as database IDs and timestamps, may be useful for identification or linking records but are not necessarily direct biological measurements.

## Data Types and Dataset Structure

The structure of the dataset is examined to identify the data type of each variable and the number of non-null values available. This will help identify potential data quality issues that require further investigation.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8915 entries, 0 to 8914
Data columns (total 33 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id                               8915 non-null   int64  
 1   ordering                         8915 non-null   float64
 2   family_name_cache                8915 non-null   object 
 3   species_name_cache               8915 non-null   object 
 4   membrane_name_cache              8915 non-null   object 
 5   name                             8915 non-null   object 
 6   description                      0 non-null      float64
 7   comments                         1202 non-null   object 
 8   pdbid                            8915 non-null   object 
 9   resolution                       8879 non-null   object 
 10  topology_subunit                 6653 non-null   object 
 11  topology_show_in                 8915 non-null   bool   
 12  thickness           

### Initial observations on dataset structure

The dataset contains a mixture of numerical, categorical, Boolean, and text variables. Several potential data quality issues are visible from the initial inspection.

- `description` and `interpro` contain no non-null values.
- `tau` and `verification` contain data for only a small proportion of the 8,915 records.
- `comments` and `topology_subunit` also contain substantial missing data.
- `resolution` contains 8,879 non-null values and is stored as an `object` rather than a numerical data type, which requires further investigation.
- `uniprotcode` is available for most, but not all, records.
- `created_at` and `updated_at` are stored as `object` values rather than datetime values.

These observations will be investigated further before making any cleaning decisions.

## Numerical Summary

Summary statistics are examined for the numerical variables to understand their ranges, central tendencies, and potential unusual values before data cleaning.

In [6]:
df.describe()

,id,ordering,description,thickness,thicknesserror,subunit_segments,tilt,tilterror,gibbs,tau,membrane_id,species_id,family_id,superfamily_id,classtype_id,type_id,secondary_representations_count,structure_subunits_count,citations_count,interpro
count,8915.000000,8915.000000,0.0,8915.000000,8880.000000,8915.000000,8915.000000,8886.000000,8915.000000,150.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,0.0
mean,4976.208749,4458.000000,NaN,22.218912,1.242827,14.132698,23.628828,3.444182,-89.141963,196.933333,7.729557,168.035895,402.484577,115.438811,2.869546,1.360067,0.892653,3.019966,0.039596,NaN
std,3127.510996,2573.683158,NaN,12.084948,2.387358,18.642101,30.735730,12.673102,88.521995,161.436283,6.728683,263.549342,369.247487,144.125066,3.019443,0.635339,4.620441,5.572643,0.347920,NaN
min,1.000000,1.000000,NaN,0.000000,-3.000000,0.000000,0.000000,-4.000000,-691.000000,60.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,NaN
25%,2291.500000,2229.500000,NaN,6.500000,0.600000,0.000000,1.000000,0.000000,-126.900000,90.000000,4.000000,14.000000,60.000000,8.000000,1.000000,1.000000,0.000000,0.000000,0.000000,NaN
50%,4576.000000,4458.000000,NaN,29.600000,1.000000,10.000000,7.000000,1.000000,-71.500000,110.000000,4.000000,36.000000,278.000000,50.000000,1.000000,1.000000,0.000000,1.000000,0.000000,NaN
75%,8103.500000,6686.500000,NaN,31.000000,1.500000,21.000000,44.000000,4.000000,-9.600000,250.000000,9.000000,213.000000,724.000000,179.000000,4.000000,2.000000,0.000000,4.000000,0.000000,NaN
max,10363.000000,8915.000000,NaN,41.800000,180.000000,389.000000,91.000000,1039.000000,165.500000,600.000000,24.000000,1150.000000,1240.000000,607.000000,11.000000,3.000000,132.000000,70.000000,11.000000,NaN


### Initial observations on numerical variables

The numerical summary shows substantial variation across several structural variables. Membrane thickness has a median of 29.6, while the number of subunit segments ranges from 0 to 389. Gibbs energy also shows a wide range of values.

Some variables contain potentially unusual values. For example, `thicknesserror` and `tilterror` have maximum values substantially higher than their upper quartiles. These values should be investigated before deciding whether they represent valid observations or data quality issues.

The `description` and `interpro` columns contain no numerical observations, while `tau` contains only 150 observations. Their usefulness will therefore require further assessment during data cleaning.

The `resolution` variable is absent from the numerical summary because it is currently stored as an `object` data type. As structural resolution is expected to represent a numerical measurement, the contents of this column will be investigated during data cleaning.

# Section 2

# Section 2: Data Cleaning
The raw dataset will be assessed for missing values, duplicate records, inappropriate data types, and variables with limited analytical value. Cleaning decisions will be made based on the characteristics of the data and the objectives of the analysis.

## Missing Values

Missing values are examined to determine the completeness of each variable and to identify columns that may require removal, transformation, or further investigation.

In [8]:
df.isnull().sum()

id                                    0
ordering                              0
family_name_cache                     0
species_name_cache                    0
membrane_name_cache                   0
name                                  0
description                        8915
comments                           7713
pdbid                                 0
resolution                           36
topology_subunit                   2262
topology_show_in                      0
thickness                             0
thicknesserror                       35
subunit_segments                      0
tilt                                  0
tilterror                            29
gibbs                                 0
tau                                8765
verification                       8693
membrane_id                           0
species_id                            0
family_id                             0
superfamily_id                        0
classtype_id                          0


---

In [9]:
missing_values = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_values

,Missing_Count,Missing_Percentage
id,0,0.00
ordering,0,0.00
family_name_cache,0,0.00
species_name_cache,0,0.00
membrane_name_cache,0,0.00
name,0,0.00
description,8915,100.00
comments,7713,86.52
pdbid,0,0.00
resolution,36,0.40


### Missing Values Observations

The analysis shows that missing data are not distributed equally across the dataset.

- `description` and `interpro` are completely empty, with 100% missing values.
- `tau` and `verification` are almost entirely missing, with 98.32% and 97.51% missing values respectively.
- `comments` contains 86.52% missing values.
- `topology_subunit` contains 25.37% missing values.
- `uniprotcode` contains 3.75% missing values.
- The structural variables `resolution`, `thicknesserror`, and `tilterror` contain less than 0.5% missing data.

Different levels of missingness require different treatment, and the biological or analytical relevance of each variable should be considered before removing data.

## Removing Empty Columns

The `description` and `interpro` columns contain no observations and therefore provide no information for the analysis. They will be removed from the working dataset.

In [12]:
df_clean = df.copy()

df_clean = df_clean.drop(columns=["description", "interpro"])

df_clean.shape

(8915, 31)

The two completely empty columns were removed, reducing the dataset from 33 to 31 variables while retaining all 8,915 protein records.

## Investigation of Sparse Columns

Several columns contain a high proportion of missing values. Before deciding whether to remove them, the available values are inspected to determine whether they provide useful information for the objectives of this analysis.

In [16]:
df_clean[["tau", "verification", "comments"]].count()





tau              150
verification     222
comments        1202
dtype: int64

In [17]:
df_clean["tau"].dropna().head(10)

33      80.0
109    130.0
189    250.0
190    150.0
191     80.0
192    120.0
193     80.0
194    110.0
195    270.0
196    100.0
Name: tau, dtype: float64

In [18]:
df_clean["verification"].dropna().head(10)



0     Four interfacial Trp residues of OmpA are loca...
1     Locations of the hydrophobic boundaries are co...
8     Membrane boundary planes of monomeric form (1q...
11    Results are consistent with experimental hydro...
19    Average tilt of TM beta-strands (38Â°) is slig...
20    Locations of hydrophobic boundaries are consis...
22    Locations of hydrophobic boundaries are consis...
27    Several Trp residues of alpha-hemolysin are si...
28    The calculated intrinsic hydrophobic thickness...
34    Calculated membrane core boundaries of MscL se...
Name: verification, dtype: object

In [19]:
df_clean["comments"].dropna().head(10)

0     OmpA is required for the action of colicins K ...
1     OmpX from Escherichia coli promotes adhesion t...
2     Pathogenic Neisseria spp. possess a repertoire...
6     Neisseria species specific OpcA proteins play ...
19    This receptor binds the ferrichrome-iron ligan...
22    Involved in the active translocation of vitami...
33    This is an open or another expanded state of t...
41    Part of the ABC transporter complex btuCDF inv...
47    Loop to helix transition in 50-residue N-termi...
55    The calculations are conducted for one half of...
Name: comments, dtype: object

### Decision on Sparse Variables

Inspection of the available values showed that the sparse columns contain meaningful information, but they are not suitable for the planned structured analysis.

- `tau` contains numerical information for only 150 of 8,915 records (1.68%), providing insufficient coverage for dataset-wide comparisons.
- `verification` contains scientifically relevant descriptive information, but is available for only 222 records (2.49%) and consists of unstructured text.
- `comments` contains useful biological and structural annotations, but is available for only 1,202 records (13.48%) and is also unstructured text.

As the project focuses on structured quantitative and categorical analysis rather than text analysis, these three variables will not be included in the analytical dataset.

In [20]:
df_clean = df_clean.drop(
    columns=["tau", "verification", "comments"]
)

df_clean.shape

(8915, 28)

## Remaining Missing Values

After removing the empty and highly sparse variables that are outside the scope of the analysis, the remaining dataset is reassessed to identify missing values that still require consideration.

In [21]:
remaining_missing = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage": (df_clean.isnull().sum() / len(df_clean) * 100).round(2)
})

remaining_missing[remaining_missing["Missing_Count"] > 0]

,Missing_Count,Missing_Percentage
resolution,36,0.40
topology_subunit,2262,25.37
thicknesserror,35,0.39
tilterror,29,0.33
uniprotcode,334,3.75


### Observation

Five variables still contain missing values. The amount of missing data varies considerably, from less than 0.5% for `resolution`, `thicknesserror`, and `tilterror`, to 25.37% for `topology_subunit`.

These variables will be investigated individually because their analytical importance and the reasons for missing values may differ.

## Investigation of the Resolution Variable

Structural resolution is expected to be numerical, but the initial dataset inspection showed that `resolution` is stored as an object data type. The values are therefore inspected before any conversion or treatment of missing data is performed.

In [22]:
df_clean["resolution"].head(20)

0     1.65
1      1.9
2     2.55
3     1.65
4     1.85
5      1.5
6     2.03
7      2.6
8      2.1
9     3.01
10     1.9
11     2.0
12     3.2
13    1.45
14     1.8
15     1.9
16     2.4
17     2.4
18     2.4
19     2.5
Name: resolution, dtype: object

In [23]:
df_clean["resolution"].unique()

array(['1.65', '1.9', '2.55', '1.85', '1.5', '2.03', '2.6', '2.1', '3.01',
       '2.0', '3.2', '1.45', '1.8', '2.4', '2.5', '2.73', '2.56', '1.89',
       '2.51', '2.8', '3.0', '3.7', '3.5', '1.4', '3.3', '2.72', '2.2',
       '2.3', '2.45', '1.6', '2.7', '6.2 EM', '3.65', '2.9', 'NMR',
       '1.47', '1.93', '3.4', '3.1', '2.35', '1.9 EM', '2.24', '4.00',
       '2.65', '3.02', '1.75', '2.08', '1.83', '1.2', '2.19', '1.41',
       '1.7', '1.04', '1.15', '3.45', '3.54 EM', '1.72', '3.9', '1.68',
       '3.3 FD', '5.0 FD', '3.1 FD', '2.4 FD', '1.74', '1.78', '2.05',
       '3.2 EM', '1.3', '1.25', '1.46', '2.59', '1.42', '9.6 EM', '2.76',
       '2.13', '1.61', '2.18', '0.85', '1.79', '1.1', '0.97', '1.97',
       '1.36', '0.99', '2.15', '1.71', '1.0', '2.27', nan, '3.8', '3.82',
       '0.98', '3.06', '2.57', '1.38', '2.31', '2.39', '2.78', '1.16',
       '2.46', '1.55', '3.6', '1.82', '2.91', '3.52', '3.88', '1.57',
       '2.02', '2.54', '2.01', '2.95', '2.06', '0.54', '0.9', '0.95'

### Resolution Data Format

Inspection of the unique values shows that `resolution` contains a mixture of numerical values and text annotations such as `EM`, `EC`, `FD`, `NMR`, and `ND`. Some records also contain additional whitespace.

This mixed formatting explains why Pandas imported the column as an `object`. The different formats will be investigated before converting the resolution values to a numerical data type.

In [24]:
resolution_numeric = pd.to_numeric(
    df_clean["resolution"],
    errors="coerce"
)

df_clean.loc[
    resolution_numeric.isna() & df_clean["resolution"].notna(),
    ["pdbid", "name", "resolution"]
].head(20)

,pdbid,name,resolution
56,"=""4aq9""","Nicotinic acetylcholine receptor, partially op...",6.2 EM
66,"=""1a91""","F0 ATP synthase, subunit c",NMR
67,"=""1c17""",F0 ATP synthase,NMR
78,"=""2jo1""","Na,K-ATPase regulatory protein FXYD1 (phosphol...",NMR
87,"=""2b6o""",Aquaporin-0,1.9 EM
91,"=""1afo""",Glycophorin A,NMR
98,"=""1grm""","Gramicidin A, head-to-head dimer, right-handed",NMR
100,"=""1bh4""",Circulin A,NMR
101,"=""1myn""",Drosomycin,NMR
105,"=""1z65""",N-terminal helix of prion-like protein doppel,NMR


### Resolution Format Observation

The inspection confirms that the `resolution` column combines numerical resolution values with additional text annotations. For example, values such as `6.2 EM` contain both a numerical resolution and an experimental annotation, while entries such as `NMR` contain no numerical resolution value.

Therefore, directly converting the existing column to a numerical data type would result in the loss of valid numerical information. The numerical component should first be extracted before conversion.

In [26]:
df_clean["resolution_numeric"] = (
    df_clean["resolution"]
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

df_clean[
    ["pdbid", "resolution", "resolution_numeric"]
].head(100)

,pdbid,resolution,resolution_numeric
0,"=""1qjp""",1.65,1.65
1,"=""1qj8""",1.9,1.90
2,"=""1p4t""",2.55,2.55
3,"=""3lbw""",1.65,1.65
4,"=""2x55""",1.85,1.85
...,...,...,...
95,"=""1t5s""",2.6,2.60
96,"=""2zbd""",2.4,2.40
97,"=""1p49""",2.6,2.60
98,"=""1grm""",NMR,NaN


### Resolution Conversion Result

The numerical component of the `resolution` column was successfully extracted into a new variable, `resolution_numeric`.

Entries containing numerical values were converted correctly, including values that also contained text annotations. Entries without a numerical resolution, such as `NMR`, resulted in missing values in the new numerical column.

The original `resolution` column was retained to preserve the source information.

In [27]:
df_clean["resolution_numeric"].isnull().sum()

1339

In [28]:
(
    df_clean["resolution_numeric"].isnull().sum()
    / len(df_clean)
    * 100
).round(2)

15.02

### Missing Numerical Resolution

After extracting the numerical component, 1,339 records (15.02%) do not contain a usable numerical resolution value.

This is substantially higher than the 36 values originally identified as missing because some populated entries contain only text annotations, such as `NMR`, rather than a numerical resolution.

These records will be retained in the dataset because they may still contain useful information for other analyses. They can be excluded only when an analysis specifically requires numerical resolution.

In [29]:
df_clean["resolution_numeric"].describe()

count    7576.000000
mean        3.064163
std         1.262766
min         0.540000
25%         2.440000
50%         3.000000
75%         3.500000
max        37.000000
Name: resolution_numeric, dtype: float64

### Resolution Distribution

The extracted numerical resolution values are available for 7,576 protein records. The median resolution is 3.0, while 50% of the observations lie between 2.44 and 3.50.

The maximum value of 37.0 is substantially higher than the upper quartile and may represent an extreme but valid observation. Extreme resolution values will therefore be inspected before deciding whether any should be removed.

In [30]:
df_clean[
    ["pdbid", "name", "resolution", "resolution_numeric"]
].sort_values(
    by="resolution_numeric",
    ascending=False
).head(20)

,pdbid,name,resolution,resolution_numeric
1945,"=""4b2q""","F1F0 ATP synthase, structure 2",37.0 EM,37.0
3047,"=""5lcb""","Bacteriochlorophyll c-binding protein, complex...",26.5 EM,26.5
2174,"=""3j41""","Aquaporin-0, complex with calmodulin",25.0 EM,25.0
3138,"=""1k4r""","Envelope glycoprotein, chimeric",24.0 EM,24.0
1305,"=""2ybb""",Respiratory complex I,19.0 EM,19.0
3474,"=""5xti""",Mitochondrial respiratory supercomplex I2-III2...,17.4 EM,17.4
6898,"=""7o01""",Photosystem I,17.1 EM,17.1
6158,"=""4ckh""","ACAP1, tetramer",17.0 EM,17.0
2197,"=""3j2s""","Coagulation factor VIII, light chain, structure 1",15.0 EM,15.0
3230,"=""5mg3""",Holo-translocon,14.0 EM,14.0


### Resolution Outlier Assessment

Inspection of the highest numerical resolution values showed that the extreme observations correspond to entries labelled `EM` in the original dataset. For example, the maximum value of 37.0 was derived from an original entry of `37.0 EM`.

These observations therefore appear to represent genuine values in the source dataset rather than errors introduced during numerical extraction. Although they are statistical outliers, they will be retained because there is no evidence that they are invalid.

If resolution is used in later visualisations, the influence of these extreme values will be considered when interpreting the results.

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [7]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)